In [ ]:
# SETUP

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Re-install LLaMA Factory and Unsloth
import os
if not os.path.exists('/content/LLaMA-Factory'):
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
    !pip install -e LLaMA-Factory/[torch,metrics,bitsandbytes,unsloth]

!pip install unsloth
!pip install unsloth "datasets<=4.0.0"
!pip install torch transformers peft accelerate
# 3. Quick Path Setup
%cd /content/LLaMA-Factory
import sys
sys.path.append(os.path.join(os.getcwd(), "src"))

In [ ]:
# TO CLONE REPOS AND CREATE DATASETS

import os
import re
import json
import subprocess

REPO_URL = "https://github.com/zephyrproject-rtos/zephyr.git"
PROJECT_NAME = "Zephyr_Dataset"
OUTPUT_FILE = f"/content/drive/MyDrive/{PROJECT_NAME}.json"


def clone_repo(url):
    repo_name = url.split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        print(f"Cloning {repo_name}...")
        subprocess.run(["git", "clone", "--depth", "1", url])
    return repo_name

def extract_functions(repo_path):
    dataset = []
    # Regex to find C functions
    func_pattern = re.compile(r'(\w+[\s\*]+)(\w+)\s*\(([^)]*)\)\s*\{([\s\S]*?)\}', re.MULTILINE)

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith((".c", ".h")):
                path = os.path.join(root, file)
                try:
                    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                        matches = func_pattern.findall(content)
                        for match in matches:
                            ret_type, name, args, body = match
                            dataset.append({
                                "instruction": f"Implement the C function '{name}' for {PROJECT_NAME}.",
                                "input": f"Return Type: {ret_type.strip()}\nArguments: {args.strip()}",
                                "output": f"{ret_type}{name}({args}) {{{body}}}"
                            })
                except Exception as e:
                    continue
    return dataset

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone and Process
repo_dir = clone_repo(REPO_URL)
print("Extracting functions")
data = extract_functions(repo_dir)

# 3. Save to JSON
with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=2)

In [ ]:
# TO MERGE DATASETS

import json
import os

os.system("rm -rf ~/.cache/huggingface/datasets/*")

base_dir = "/content/drive/MyDrive"
master_dataset = []

# 2. Process the standard JSON files
standard_json_files = [
    f"{base_dir}/misra_c_2012_alpaca.json",
    f"{base_dir}/Zephyr_Dataset.json",
    f"{base_dir}/FreeRTOS_Dataset.json",
    f"{base_dir}/STMCubeHAL_Dataset.json"
]

for f_path in standard_json_files:
    if os.path.exists(f_path):
        with open(f_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for row in data:
                master_dataset.append({
                    "instruction": str(row.get("instruction", "")),
                    "input": str(row.get("input", "")),
                    "output": str(row.get("output", ""))
                })
    else:
        print(f"Missing: {f_path}")

# 3. Process the JSONL file
jsonl_path = f"{base_dir}/sof_le_1000_40k.jsonl"
if os.path.exists(jsonl_path):
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                master_dataset.append({
                    "instruction": str(row.get("question", "")),
                    "input": "",
                    "output": str(row.get("answer", ""))
                })
else:
    print(f"Missing: {jsonl_path}")

# 4. Save the unified Master File
master_path = f"{base_dir}/project_master.json"
with open(master_path, 'w', encoding='utf-8') as f:
    json.dump(master_dataset, f, indent=2)

print(f"\nTotal examples: {len(master_dataset)}")

# 5. Lock the LLaMA Factory Registry to ONLY the master file
registry_path = '/content/LLaMA-Factory/data/dataset_info.json'

strict_registry = {
    "project_master": {
        "file_name": master_path
    }
}

with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(strict_registry, f, indent=2)

In [ ]:
# To clear data folder and keep only required json file

import json

registry_path = '/content/LLaMA-Factory/data/dataset_info.json'

# Lock the registry exclusively to your existing master file
strict_registry = {
    "project_master": {
        "file_name": "/content/drive/MyDrive/project_master.json"
    }
}

with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(strict_registry, f, indent=2)

In [ ]:
# TO LAUNCH LLAMA FACTORY WEB-UI

# 1. Move to the correct folder
%cd /content/LLaMA-Factory

# 2. Add the source code to the path and launch the UI
!export PYTHONPATH=$PYTHONPATH:$(pwd)/src && GRADIO_SHARE=True llamafactory-cli webui

In [ ]:
# TO CLEAR GPU CACHE INCASE OF ISSUES

import torch
import gc
gc.collect()
torch.cuda.empty_cache()